# HTML and Webscraping

In [ ]:
import pandas as pd

## Scraping an HTML table

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2025). We'll use Beautiful Soup to scrape information from this table.

1\. Read in the HTML from the URL using the `requests` library.

In [2]:
# YOUR CODE HERE
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"

response = requests.get(url, headers ={"User-Agent": "Mozilla/5.0"},)

2\. Use Beautiful Soup to parse this string into a tree called `soup`

In [3]:
# YOUR CODE HERE
soup = BeautifulSoup(response.text, "html.parser")

3\. Determine how many tables are in `soup`. (Hint: use `find_all("table")`.)

In [4]:
# YOUR CODE HERE
tables = soup.find_all("table")
len(tables)

10

4\. There are several tables included in `soup`, so we need to narrow it down. Go to the cities table Wikipedia page and "Inspect" it. What are the attributes (class, style) of this table?



In [12]:
# YOUR CODE HERE
cities = soup.find_all("table",
attrs={"class":"sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center jquery-tablesorter", "style":"text-align:right"})


In [13]:
len(cities)

0

5\. You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center" style="text-align:right">
```

How many tables in `soup` have these attributes?

In [14]:
# YOUR CODE HERE
len(soup.find_all("table", attrs={"class":"sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center", }))

1

6\. There should only be 1 table of this type, so we just need to select it. The following code finds all tables with the desired attributes and then selects the first (only) one to store as `table`. (You just need to run this.)

In [15]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

7\. Our goal is now to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for:

- city
- state
- population (2025 estimate)
- 2020 land area (sq mi).

First, let's just see how to scrape the information for New York City. Starting from `table` create an object, called `city`, that contains the information just for New York City.

Hints: Inspect the source; what kind of tag represents each row? Find all tags of this type in `table` and select the first one that corresponds to a city. Note that the first 3 rows of the table are headers.

In [20]:
# YOUR CODE HERE
city = table.find_all("tr")[3]
city

<tr id="mw0g">
<td id="mw0w" style="background-color:#cfecec"><a href="https://en.wikipedia.org/wiki/New_York_City" id="mw1A" rel="mw:WikiLink" title="New York City">New York</a><sup about="#mwt57" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-5"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"1":{"wt":"Since 1898, the [[New York City|City of New York]], New York, has comprised [[borough of New York City|five boroughs]] with [[consolidated city-county|consolidated borough–county governments]] (2025 population estimates):\n* [[Brooklyn|The Borough of Brooklyn]] and [[Brooklyn|Kings County]]\n: (pop. 2,653,963)\n* [[Queens|The Borough of Queens]] and [[Queens|Queens County]]\n: (pop. 2,358,182)\n* [[Manhattan|The Borough of Manhattan]] and [[Manhattan|New York County]]\n: (pop. 1,664,862)\n* [[The Bronx|The Borough of the Bronx]] and [[The Bronx|Bronx County]]\n: (pop. 1,406,332)\n

In [21]:

ny = table.find_all("tr", attrs= {"id": "mw0g"})[0]
ny

<tr id="mw0g">
<td id="mw0w" style="background-color:#cfecec"><a href="https://en.wikipedia.org/wiki/New_York_City" id="mw1A" rel="mw:WikiLink" title="New York City">New York</a><sup about="#mwt57" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-5"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"1":{"wt":"Since 1898, the [[New York City|City of New York]], New York, has comprised [[borough of New York City|five boroughs]] with [[consolidated city-county|consolidated borough–county governments]] (2025 population estimates):\n* [[Brooklyn|The Borough of Brooklyn]] and [[Brooklyn|Kings County]]\n: (pop. 2,653,963)\n* [[Queens|The Borough of Queens]] and [[Queens|Queens County]]\n: (pop. 2,358,182)\n* [[Manhattan|The Borough of Manhattan]] and [[Manhattan|New York County]]\n: (pop. 1,664,862)\n* [[The Bronx|The Borough of the Bronx]] and [[The Bronx|Bronx County]]\n: (pop. 1,406,332)\n

8\. Starting with `city` extract the city's name and store it as `name`.

Hints: Inspect the source; what tag represents the cells within a row? Find all tags of this type and extract the text corresponding to the cell with the city's name.

In [27]:
# YOUR CODE HERE
name = city.find_all("td")
name

[<td id="mw0w" style="background-color:#cfecec"><a href="https://en.wikipedia.org/wiki/New_York_City" id="mw1A" rel="mw:WikiLink" title="New York City">New York</a><sup about="#mwt57" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-5"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"1":{"wt":"Since 1898, the [[New York City|City of New York]], New York, has comprised [[borough of New York City|five boroughs]] with [[consolidated city-county|consolidated borough–county governments]] (2025 population estimates):\n* [[Brooklyn|The Borough of Brooklyn]] and [[Brooklyn|Kings County]]\n: (pop. 2,653,963)\n* [[Queens|The Borough of Queens]] and [[Queens|Queens County]]\n: (pop. 2,358,182)\n* [[Manhattan|The Borough of Manhattan]] and [[Manhattan|New York County]]\n: (pop. 1,664,862)\n* [[The Bronx|The Borough of the Bronx]] and [[The Bronx|Bronx County]]\n: (pop. 1,406,332)\n* [[Staten Isl

In [41]:
name = city.find_all("td")[0].text
name

'New York[c]'

9\. Extract the city's state and store it as `state`.

In [43]:
# YOUR CODE HERE
state = city.find_all("td")[1].text
state

'NY'

10\. Extract the city's population and store is as `population`.

In [37]:
# YOUR CODE HERE
population = city.find_all("td")[2].text
population

'8,584,629'

11\. Extract the city's area and store is as `area`.

In [40]:
# YOUR CODE HERE
area = city.find_all("td")[5].text
area

'300.5'

12\. Now put the steps for a single city into a loop to extract the information for all cities in `table` and create a data frame.

Hints:
- Start with an empty list named `rows`
- Write a loop that starts `for city in ...` and replace `...` code that finds all the table rows. (Use what you did in part 7, but don't just select one row. Select all rows except for the 3 header rows.)
- Use your code from 8-11 to extract the information for the city
- And append it to `rows` as "name", "state", "population", "area"
- Convert `rows` into a Pandas data frame. You should obtain a data frame with 348 rows and 4 columns.


In [58]:
import pandas as pd

rows = []

for city_row in table.find_all("tr")[3:]:
  cells = city_row.find_all("td")

  name = cells[0].text
  state = cells[1].text
  population = cells[2]
  area = cells[5]

  rows.append({
        "city": name,
        "state": state,
        "population": (population),
        "area": (area)
    })

df = pd.DataFrame(rows)
df

,city,state,population,area
0,New York[c],NY,"[8,584,629]",[300.5]
1,Los Angeles,CA,"[3,869,089]",[469.5]
2,Chicago,IL,"[2,731,585]",[227.7]
3,Houston,TX,"[2,397,315]",[640.4]
4,Phoenix,AZ,"[1,665,481]",[518.0]
...,...,...,...,...
343,San Angelo,TX,"[100,640]",[59.7]
344,Edmond,OK,"[100,479]",[84.6]
345,Davenport,IA,"[100,358]",[63.8]
346,Deltona,FL,"[100,267]",[37.3]


13\. Use the Pandas command `pd.read_html` can be used to scrape the table from the webpage. Note: `read_html` will return all the tables, so you will need to narrow your request using attributes. You don't need to worry about selecting columns; just scrape the whole table.

In [71]:
# YOUR CODE HERE
respone = requests.get("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", headers ={"User-Agent": "Mozilla/5.0"},)

In [72]:
import pandas as pd
df = pd.read_html(respone.text)

/tmp/ipykernel_2449/1991424404.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(respone.text)


## Scraping from multiple webpages

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/. Notice that the information is spread over many pages.

1\. Scrape the information from the first page with Beatiful Soup.

In [74]:
# YOUR CODE HERE
url = "https://www.scrapethissite.com/pages/forms/"

rsp = requests.get(url, headers = {"User-Agent": "Mozilla/5.0"},)
sop = BeautifulSoup(rsp.text, "html.parser")

2\. Find the main table on this page and store it as `table`.

In [76]:
# YOUR CODE HERE
tables = sop.find_all("table")
tables

[<table class="table">
 <tr>
 <th>
                             Team Name
                         </th>
 <th>
                             Year
                         </th>
 <th>
                             Wins
                         </th>
 <th>
                             Losses
                         </th>
 <th>
                             OT Losses
                         </th>
 <th>
                             Win %
                         </th>
 <th>
                             Goals For (GF)
                         </th>
 <th>
                             Goals Against (GA)
                         </th>
 <th>
                             + / -
                         </th>
 </tr>
 <tr class="team">
 <td class="name">
                             Boston Bruins
                         </td>
 <td class="year">
                             1990
                         </td>
 <td class="wins">
                             44
                         </td>
 <td clas

In [85]:
wins = tables[0].find_all("td")[2].text.strip()
wins

'44'

3\. Extract the information from the cells of this table into a Pandas data frame.

In [91]:
rows = []

table = tables[0] # Assign the correct table for the hockey stats

for row in table.find_all("tr")[1:]:
  cells = row.find_all("td")

  team_name = cells[0].text.strip()
  year = cells[1].text.strip()
  wins = cells[2].text.strip()
  loss = cells[3].text.strip()
  OT_loss = cells[4].text.strip()
  win_pct = cells[5].text.strip()
  goals_for_gf = cells[6].text.strip()
  goals_against_ga = cells[7].text.strip()
  pos_negative = cells[8].text.strip()

  rows.append({
        "team_name": team_name,
        "year": year,
        "wins": wins,
        "loss": loss,
        "OT_loss": OT_loss,
        "win_pct": win_pct,
        "goals_for_gf": goals_for_gf,
        "goals_against_ga": goals_against_ga,
        "pos_negative": pos_negative
    })

df = pd.DataFrame(rows)
df

,team_name,year,wins,loss,OT_loss,win_pct,goals_for_gf,goals_against_ga,pos_negative
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25
5,Edmonton Oilers,1990,37,37,,0.463,272,272,0
6,Hartford Whalers,1990,31,38,,0.388,238,276,-38
7,Los Angeles Kings,1990,46,24,,0.575,340,254,86
8,Minnesota North Stars,1990,27,39,,0.338,256,266,-10
9,Montreal Canadiens,1990,39,30,,0.487,273,249,24


In [ ]:
# YOUR CODE HERE

4\. But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We could switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [94]:
pagination = soup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [95]:
for link in links:
  print(link.attrs["href"])

/pages/forms/?page_num=1
/pages/forms/?page_num=2
/pages/forms/?page_num=3
/pages/forms/?page_num=4
/pages/forms/?page_num=5
/pages/forms/?page_num=6
/pages/forms/?page_num=7
/pages/forms/?page_num=8
/pages/forms/?page_num=9
/pages/forms/?page_num=10
/pages/forms/?page_num=11
/pages/forms/?page_num=12
/pages/forms/?page_num=13
/pages/forms/?page_num=14
/pages/forms/?page_num=15
/pages/forms/?page_num=16
/pages/forms/?page_num=17
/pages/forms/?page_num=18
/pages/forms/?page_num=19
/pages/forms/?page_num=20
/pages/forms/?page_num=21
/pages/forms/?page_num=22
/pages/forms/?page_num=23
/pages/forms/?page_num=24
/pages/forms/?page_num=1


Now we can loop over `links` to make a request to the url for each page and scrape the data into a table similar to what we did for the first page. Write such a loop to extract the data and create a Pandas data frame.


A few technicalities:

- You might need to skip the "previous" and "next" buttons
- So you don't keep repeating headers, you will want to skip rows that don't represent teams.

In [96]:
import time

all_rows = []
seen_pages = set()

for link in links:
    href = link.attrs["href"]

    # skip "previous" and duplicate/blank hrefs
    if "page_num" not in href:
        continue
    if href in seen_pages:
        continue
    seen_pages.add(href)

    page_url = "https://www.scrapethissite.com" + href
    rsp = requests.get(page_url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(rsp.text, "html.parser")

    table = soup.find("table")
    rows = table.find_all("tr")

    for row in rows:
        cells = row.find_all("td")

        # skip header row / any row that isn't a full team row
        if len(cells) < 9:
            continue

        team_name = cells[0].text.strip()
        year = cells[1].text.strip()
        wins = cells[2].text.strip()
        loss = cells[3].text.strip()
        OT_loss = cells[4].text.strip()
        win_pct = cells[5].text.strip()
        goals_for_gf = cells[6].text.strip()
        goals_against_ga = cells[7].text.strip()
        pos_negative = cells[8].text.strip()

        all_rows.append({
            "team_name": team_name,
            "year": year,
            "wins": wins,
            "loss": loss,
            "OT_loss": OT_loss,
            "win_pct": win_pct,
            "goals_for_gf": goals_for_gf,
            "goals_against_ga": goals_against_ga,
            "pos_negative": pos_negative
        })

    time.sleep(0.5)  # be polite to the server

df_all = pd.DataFrame(all_rows)
df_all

,team_name,year,wins,loss,OT_loss,win_pct,goals_for_gf,goals_against_ga,pos_negative
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25
...,...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9,0.622,249,198,51
580,Washington Capitals,2011,42,32,8,0.512,222,230,-8


In [ ]:
# YOUR CODE HERE
